# Cross-Dataset Validation — Le2i Fall Detection

**Purpose:** Evaluate GhostNeck-YOLOv8n on the Le2i dataset to assess generalisation of fall detection beyond the Strathclyde training distribution.

**Dataset:** Le2i Object Detection (Roboflow Universe, 2022) — 3,010 images, `fall` class only.  
The class label is remapped from `fall` → `fall_detected` to match the model's training vocabulary.

**Note:** Sitting and walking classes are excluded as Le2i does not provide corresponding annotations. Cross-dataset validation is therefore scoped to fall detection performance only.

---
## Step 1 — Environment Setup

In [ ]:
!pip install ultralytics -q

import torch
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 59.1 MB/s eta 0:00:00
PyTorch : 2.10.0+cu128
CUDA    : True
GPU     : Tesla T4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Google Drive mounted.")

Mounted at /content/drive
Google Drive mounted.


---
## Step 2 — Extract Le2i Dataset from Google Drive

In [ ]:
import os, zipfile
from pathlib import Path
import shutil
shutil.rmtree("/content/le2i")
print("Cleared.")

# ============================================================
# CONFIG — only change these two paths
# ============================================================
DRIVE_ZIP   = "/content/drive/MyDrive/Le2i.v1i.yolov8.zip"
GHOST_PT    = "/content/drive/MyDrive/fall_detection_ghostneck_results_v2/best.pt"
EXTRACT_DIR = "/content/le2i"
# ============================================================

# Verify zip exists (also try without .zip extension)
if not os.path.exists(DRIVE_ZIP):
    alt = DRIVE_ZIP.replace(".zip", "")
    if os.path.exists(alt):
        DRIVE_ZIP = alt
    else:
        print("Zip not found. Files in your Drive root:")
        for f in sorted(os.listdir("/content/drive/MyDrive/")):
            if any(k in f.lower() for k in ["le2i", "fall"]):
                print(f"  {f}  <-- possible match")
            else:
                print(f"  {f}")
        raise FileNotFoundError("Update DRIVE_ZIP in the CONFIG section above.")

print(f"Found : {DRIVE_ZIP}  ({os.path.getsize(DRIVE_ZIP)/1024**2:.1f} MB)")

# Extract (skip if already done)
if os.path.exists(EXTRACT_DIR):
    print("Extract directory already exists — skipping extraction.")
else:
    print(f"Extracting to {EXTRACT_DIR} ...")
    with zipfile.ZipFile(DRIVE_ZIP, 'r') as zf:
        zf.extractall(EXTRACT_DIR)
    print("Done.")

# Print structure for verification
le2i_dir = Path(EXTRACT_DIR)
print("\nDataset structure:")
for root, dirs, files in os.walk(str(le2i_dir)):
    level = root.replace(str(le2i_dir), '').count(os.sep)
    if level <= 2:
        print(f"{'  '*level}{os.path.basename(root)}/ ({len(files)} files)")

Cleared.
Found : /content/drive/MyDrive/Le2i.v1i.yolov8.zip  (34.2 MB)
Extracting to /content/le2i ...
Done.

Dataset structure:
le2i/ (0 files)
  Le2i.v1i.yolov8/ (6 files)
    .idea/ (5 files)
    train/ (0 files)
    valid/ (0 files)
    test/ (0 files)


---
## Step 3 — Remap Class Label: `fall` → `fall_detected`

Le2i uses `fall` (class index 0); our model was trained on `fall_detected` (also index 0).  
Annotation `.txt` files do **not** need editing — the index is already correct.  
We only rewrite `data.yaml` to update the class name and set absolute Colab paths.

In [ ]:
import yaml

# Locate data.yaml inside the extracted folder
yaml_candidates = list(le2i_dir.rglob("data.yaml"))
if not yaml_candidates:
    raise FileNotFoundError("data.yaml not found in the extracted dataset.")

original_yaml = yaml_candidates[0]
dataset_root  = original_yaml.parent

with open(original_yaml) as f:
    config = yaml.safe_load(f)

print("Original data.yaml:")
print(f"  nc    : {config.get('nc')}")
print(f"  names : {config.get('names')}")

# Update class vocabulary to match training setup.
# Le2i only annotates fall (class 0 -> fall_detected).
# Sitting/walking are included in the name list for index consistency
# but will never appear in Le2i ground truth.
config["nc"]    = 3
config["names"] = ["fall_detected", "sitting", "walking"]
config["path"]  = str(dataset_root)

# Fix split paths to absolute Colab paths
for split in ["train", "val", "test"]:
    if split not in config:
        continue
    # Try the value as written, then as a subdirectory name
    for candidate in [
        dataset_root / config[split],
        dataset_root / split / "images",
        dataset_root / split,
        dataset_root / "valid" / "images",   # Roboflow sometimes uses 'valid'
    ]:
        if Path(candidate).exists():
            config[split] = str(candidate)
            break

LE2I_YAML = str(dataset_root / "data_remapped.yaml")
with open(LE2I_YAML, 'w') as f:
    yaml.dump(config, f, default_flow_style=False, allow_unicode=True)

print("\nRemapped data.yaml:")
print(f"  nc    : {config['nc']}")
print(f"  names : {config['names']}")
print(f"  train : {config.get('train')}")
print(f"  val   : {config.get('val')}")
print(f"  Saved : {LE2I_YAML}")

Original data.yaml:
  nc    : 1
  names : ['fall']

Remapped data.yaml:
  nc    : 3
  names : ['fall_detected', 'sitting', 'walking']
  train : /content/le2i/Le2i.v1i.yolov8/train/images
  val   : /content/le2i/Le2i.v1i.yolov8/valid/images
  Saved : /content/le2i/Le2i.v1i.yolov8/data_remapped.yaml


In [ ]:
from PIL import Image
from pathlib import Path

# 检查几张Le2i的val图片
val_imgs = list(Path("/content/le2i").rglob("valid/images/*.jpg"))[:5]
for p in val_imgs:
    img = Image.open(p)
    print(f"{p.name}: mode={img.mode}, size={img.size}")

002153_jpg.rf.3d71aa9f82d47ae41d234aae0b9d7268.jpg: mode=RGB, size=(320, 240)
002096_jpg.rf.f79a5ddc583deaaaa123dabd58a1f88a.jpg: mode=RGB, size=(320, 240)
000544_jpg.rf.27fd920af1ea6b4951bc7a5717f1b97b.jpg: mode=RGB, size=(320, 180)
002078_jpg.rf.98482941e79ed338b0fac50a5714785a.jpg: mode=RGB, size=(320, 240)
001025_jpg.rf.2be2df80defc99fb5b722595e1a966ba.jpg: mode=RGB, size=(320, 240)


In [ ]:
# 检查几个label文件的框大小
val_labels = list(Path("/content/le2i").rglob("valid/labels/*.txt"))[:5]
for p in val_labels:
    print(p.name, p.read_text().strip())

001826_jpg.rf.4be04584f055a8a76f7691196d097122.txt 0 0.4640625 0.5 0.221875 0.5666666666666667
000220_jpg.rf.31c6fabd49eeee1284af46ec23bf199d.txt 0 0.525 0.7861111111111111 0.20625 0.42777777777777776
000484_jpg.rf.324f7702b7b2f70952a5de78d5b76081.txt 0 0.5171875 0.7055555555555556 0.359375 0.43333333333333335
001261_jpg.rf.dcd00bd911221ea1f58d95d180a7afe8.txt 0 0.4625 0.7520833333333333 0.3 0.4791666666666667
000005_jpg.rf.0ebaf0abf285d0d4f5485fbc46cdde62.txt 0 0.578125 0.48125 0.1625 0.6291666666666667


In [ ]:
from pathlib import Path

for split in ["train", "valid", "test"]:
    img_dir = Path("/content/le2i/Le2i.v1i.yolov8") / split / "images"
    imgs = list(img_dir.glob("*")) if img_dir.exists() else []
    print(f"{split}: {len(imgs)} images")

train: 2411 images
valid: 301 images
test: 298 images


---
## Step 4 — Run Validation

The model was never trained on any Le2i data — this is a true zero-shot cross-dataset evaluation.

In [ ]:
from ultralytics import YOLO

if not os.path.exists(GHOST_PT):
    raise FileNotFoundError(
        f"Model weights not found: {GHOST_PT}\n"
        "Run Notebook 2 first to generate the GhostNeck weights."
    )

model = YOLO(GHOST_PT)

le2i_metrics = model.val(
    data    = LE2I_YAML,
    split   = "val",
    imgsz   = 640,
    conf    = 0.25,
    iou     = 0.5,
    verbose = True,
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.40 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
YOLOv8n-ghostneck summary (fused): 103 layers, 2,422,529 parameters, 0 gradients, 7.0 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 269.9±152.7 MB/s, size: 10.4 KB)
val: Scanning /content/le2i/Le2i.v1i.yolov8/valid/labels... 301 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 301/301 2.4Kit/s 0.1s
val: New cache created: /content/le2i/Le2i.v1i.yolov8/valid/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 19/19 3.5it/s 5.4s
                   all        301        301      0.812      0.542      0.552      0.274
   

---
## Step 5 — Results Summary

In [ ]:
p_per_class    = le2i_metrics.box.p
r_per_class    = le2i_metrics.box.r
ap50_per_class = le2i_metrics.box.ap50

fall_p  = float(p_per_class[0])    if len(p_per_class)    > 0 else 0.0
fall_r  = float(r_per_class[0])    if len(r_per_class)    > 0 else 0.0
fall_ap = float(ap50_per_class[0]) if len(ap50_per_class) > 0 else 0.0
fall_f1 = 2 * fall_p * fall_r / (fall_p + fall_r) if (fall_p + fall_r) > 0 else 0.0

print("\n" + "=" * 62)
print("Cross-Dataset Validation — Le2i (fall_detected only)")
print("=" * 62)
print(f"{'Metric':<30} {'Le2i':>10} {'Strathclyde':>14}")
print("-" * 56)
print(f"{'Precision (fall_detected)':<30} {fall_p:>10.4f} {'0.9697':>14}")
print(f"{'Recall    (fall_detected)':<30} {fall_r:>10.4f} {'0.9199':>14}")
print(f"{'F1        (fall_detected)':<30} {fall_f1:>10.4f} {'0.9444':>14}")
print(f"{'AP@0.5    (fall_detected)':<30} {fall_ap:>10.4f} {'0.9634':>14}")
print()
print("Note: Performance gap vs Strathclyde reflects domain shift")
print("      (different camera angles, lighting, and environments).")
print("      Sitting and walking are excluded — not annotated in Le2i.")


Cross-Dataset Validation — Le2i (fall_detected only)
Metric                               Le2i    Strathclyde
--------------------------------------------------------
Precision (fall_detected)          0.8118         0.9697
Recall    (fall_detected)          0.5415         0.9199
F1        (fall_detected)          0.6497         0.9444
AP@0.5    (fall_detected)          0.5516         0.9634

Note: Performance gap vs Strathclyde reflects domain shift
      (different camera angles, lighting, and environments).
      Sitting and walking are excluded — not annotated in Le2i.


In [ ]:
# Save results to Google Drive
import json, shutil

SAVE_DIR = Path("/content/drive/MyDrive/fall_detection_ghostneck_results_v2/le2i_validation")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

result_dict = {
    "dataset"           : "Le2i (Roboflow Universe, 2022)",
    "model"             : "GhostNeck-YOLOv8n (trained on Strathclyde)",
    "split"             : "val",
    "classes_evaluated" : ["fall_detected"],
    "precision"         : round(fall_p,  4),
    "recall"            : round(fall_r,  4),
    "f1"                : round(fall_f1, 4),
    "ap50"              : round(fall_ap, 4),
}

with open(SAVE_DIR / "le2i_results.json", 'w') as f:
    json.dump(result_dict, f, indent=2)

shutil.copy(LE2I_YAML, SAVE_DIR / "data_remapped.yaml")

print(f"Results saved to: {SAVE_DIR}")
print(json.dumps(result_dict, indent=2))

Results saved to: /content/drive/MyDrive/fall_detection_ghostneck_results_v2/le2i_validation
{
  "dataset": "Le2i (Roboflow Universe, 2022)",
  "model": "GhostNeck-YOLOv8n (trained on Strathclyde)",
  "split": "val",
  "classes_evaluated": [
    "fall_detected"
  ],
  "precision": 0.8118,
  "recall": 0.5415,
  "f1": 0.6497,
  "ap50": 0.5516
}
